In [0]:
# ============================================================
# NOTEBOOK : DIM_ORDER
# PURPOSE  : ORDER DIMENSION INCREMENTAL LOAD
# ============================================================

from pyspark.sql.functions import *
from delta.tables import *
import uuid

In [0]:
%run /Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_Common_Functions

In [0]:
%run "/Users/ragul.p.dev@gmail.com/fabric_incremental_project/Functions/FN_LOGGER"

Max Date updated successfully


In [0]:
# ============================================================
# GET BRONZE DATA
# ============================================================

try:

    metadata = get_metadata("orders_tbl")

    table_name = metadata["target_table"]

    source_system = metadata["source_system"]

    watermark_column = metadata["watermark_column"]

    primary_key = metadata["primary_key_column"]

    source_table = f"bronze.{table_name}"

    target_table = "silver.dim_order"

    pipeline_name = "PL_DIM_ORDER"

    pipeline_run_id = str(uuid.uuid4())

    start_time = get_current_timestamp()

    last_watermark = get_watermark(table_name)

    print(f"Last Watermark : {last_watermark}")

    bronze_df = spark.sql(f"""

        SELECT

            order_id,
            customer_id,
            order_date,
            order_status,
            shipping_address,
            shipping_city,
            shipping_state,
            shipping_country,
            total_amount,
            created_date,
            modified_date

        FROM {source_table}

        WHERE {watermark_column} > '{last_watermark}'

    """)

    bronze_df.createOrReplaceTempView(
        "vw_bronze_order"
    )

    print(f"Bronze View Created : {table_name}")

    rows_read = bronze_df.count()

    print(f"Rows Read : {rows_read}")

except Exception as e:

    print(f"Bronze View Creation Failed : {table_name}")

    raise(e)



Last Watermark : 1900-01-01 00:00:00
Bronze View Created : orders_tbl
Rows Read : 5


FN_LOGGER LOADED SUCCESSFULLY


FN_COMMON_FUNCTIONS LOADED SUCCESSFULLY


In [0]:
# ============================================================
# CREATE SILVER VIEW
# ============================================================

try:

    silver_df = spark.sql("""

        SELECT

            o.order_id,

            o.customer_id,

            c.customer_name,

            o.order_date,

            upper(o.order_status)
                AS order_status,

            initcap(o.shipping_address)
                AS shipping_address,

            initcap(o.shipping_city)
                AS shipping_city,

            initcap(o.shipping_state)
                AS shipping_state,

            upper(o.shipping_country)
                AS shipping_country,

            o.total_amount,

            CASE

                WHEN o.total_amount >= 50000
                THEN 'HIGH_VALUE'

                WHEN o.total_amount >= 10000
                THEN 'MEDIUM_VALUE'

                ELSE 'LOW_VALUE'

            END AS order_category,

            CASE

                WHEN upper(o.shipping_state) = 'TAMIL NADU'
                THEN 'SOUTH'

                WHEN upper(o.shipping_state) = 'KARNATAKA'
                THEN 'SOUTH'

                WHEN upper(o.shipping_state) = 'MAHARASHTRA'
                THEN 'WEST'

                WHEN upper(o.shipping_state) = 'DELHI'
                THEN 'NORTH'

                ELSE 'OTHER'

            END AS shipping_region,

            o.modified_date,

            sha2(
                concat_ws(
                    '|',
                    o.customer_id,
                    c.customer_name,
                    o.order_status,
                    o.shipping_city,
                    o.shipping_state,
                    o.total_amount
                ),
                256
            ) AS hash_key,

            current_timestamp()
                AS effective_start_date,

            CAST(NULL AS TIMESTAMP)
                AS effective_end_date,

            1 AS is_current,

            0 AS is_deleted

        FROM vw_bronze_order o

        LEFT JOIN silver.dim_customer c
            ON o.customer_id = c.customer_id
            AND c.is_current = 1

    """)

    silver_df.createOrReplaceTempView(
        "vw_silver_order"
    )

    print(f"Silver View Created : {table_name}")

except Exception as e:

    print(f"Silver View Creation Failed : {table_name}")

    raise(e)



Silver View Created : orders_tbl


In [0]:
# ============================================================
# CREATE TARGET TABLE
# ============================================================

try:

    spark.sql("""

        CREATE TABLE IF NOT EXISTS silver.dim_order
        (
            order_id BIGINT,
            customer_id BIGINT,
            customer_name STRING,
            order_date TIMESTAMP,
            order_status STRING,
            shipping_address STRING,
            shipping_city STRING,
            shipping_state STRING,
            shipping_country STRING,
            total_amount DOUBLE,
            order_category STRING,
            shipping_region STRING,
            modified_date TIMESTAMP,
            hash_key STRING,
            effective_start_date TIMESTAMP,
            effective_end_date TIMESTAMP,
            is_current INT,
            is_deleted INT
        )

        USING DELTA

    """)

    print(f"Target Table Created : {target_table}")

except Exception as e:

    print(f"Target Table Creation Failed : {target_table}")

    raise(e)



Target Table Created : silver.dim_order


In [0]:
# ============================================================
# MERGE LOGIC
# ============================================================

try:

    spark.sql("""

        MERGE INTO silver.dim_order AS target

        USING vw_silver_order AS source

        ON target.order_id = source.order_id
           AND target.is_current = 1

        WHEN MATCHED
             AND target.hash_key <> source.hash_key

        THEN UPDATE SET

            target.effective_end_date =
                current_timestamp(),

            target.is_current = 0

        WHEN NOT MATCHED

        THEN INSERT
        (
            order_id,
            customer_id,
            customer_name,
            order_date,
            order_status,
            shipping_address,
            shipping_city,
            shipping_state,
            shipping_country,
            total_amount,
            order_category,
            shipping_region,
            modified_date,
            hash_key,
            effective_start_date,
            effective_end_date,
            is_current,
            is_deleted
        )

        VALUES
        (
            source.order_id,
            source.customer_id,
            source.customer_name,
            source.order_date,
            source.order_status,
            source.shipping_address,
            source.shipping_city,
            source.shipping_state,
            source.shipping_country,
            source.total_amount,
            source.order_category,
            source.shipping_region,
            source.modified_date,
            source.hash_key,
            source.effective_start_date,
            source.effective_end_date,
            source.is_current,
            source.is_deleted
        )

    """)

    print(f"Merge Completed : {table_name}")

    spark.sql("""

        UPDATE silver.dim_order

        SET

            is_deleted = 1,
            is_current = 0,
            effective_end_date = current_timestamp()

        WHERE order_id NOT IN
        (
            SELECT order_id
            FROM vw_silver_order
        )

        AND is_current = 1

    """)

    print(f"Soft Delete Completed : {table_name}")

except Exception as e:

    print(f"Merge Failed : {table_name}")

    raise(e)



Merge Completed : orders_tbl
Soft Delete Completed : orders_tbl


In [0]:
# ============================================================
# UPDATE WATERMARK & AUDIT LOG
# ============================================================

try:

    max_date = get_max_date(
        bronze_df,
        watermark_column
    )

    if max_date is not None:

        update_watermark(
            table_name,
            max_date
        )

        print(f"Watermark Updated : {table_name}")

    else:

        print("No Incremental Records Found")

    rows_written = silver_df.count()

    end_time = get_current_timestamp()

    execution_time_seconds = int(
        (end_time - start_time).total_seconds()
    )

    insert_audit_log(

        pipeline_run_id,
        pipeline_name,
        source_system,
        table_name,
        start_time,
        end_time,
        rows_read,
        rows_written,
        "SUCCESS",
        execution_time_seconds

    )

    print(f"DIM_ORDER SUCCESSFULLY LOADED : {table_name}")

except Exception as e:

    insert_error_log(

        str(uuid.uuid4()),
        pipeline_run_id,
        table_name,
        source_system,
        "DIM_ORDER",
        str(e)

    )

    print(f"DIM_ORDER LOAD FAILED : {table_name}")

    raise(e)

Watermark Updated : orders_tbl
Watermark Updated : orders_tbl
Audit Log Inserted : orders_tbl
DIM_ORDER SUCCESSFULLY LOADED : orders_tbl
